In [1]:
import pandas as pd


In [14]:
bioinfo = pd.read_parquet('../data/raw/bioinfo.parquet')
ideology = pd.read_parquet('../data/raw/ideology.parquet')
terms = pd.read_parquet('../data/raw/terms.parquet')
bioinfo.info()


<class 'pandas.DataFrame'>
RangeIndex: 555 entries, 0 to 554
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   bioguide_id     555 non-null    str    
 1   Full name       555 non-null    str    
 2   Chamber         555 non-null    str    
 3   State           555 non-null    str    
 4   Party           555 non-null    str    
 5   District        439 non-null    float64
 6   birthYear       554 non-null    str    
 7   image           554 non-null    str    
 8   Office address  539 non-null    str    
 9   Phone           539 non-null    str    
 10  Website         539 non-null    str    
dtypes: float64(1), str(10)
memory usage: 166.9 KB


In [3]:
bioinfo['bioguide_id'].duplicated().value_counts()

bioguide_id
False    555
Name: count, dtype: int64

In [4]:
bioinfo.head(3).T

,0,1,2
bioguide_id,R000575,S001185,P000609
Full name,Mike Rogers,Terri A. Sewell,Gary J. Palmer
Chamber,House of Representatives,House of Representatives,House of Representatives
State,Alabama,Alabama,Alabama
Party,Republican,Democratic,Republican
District,3.0,7.0,6.0
birthYear,1958,1965,1954
image,https://www.congress.gov/img/member/116_rp_al_...,https://www.congress.gov/img/member/s001185_20...,https://www.congress.gov/img/member/p000609_20...
Office address,"2469 Rayburn House Office Building, Washington...","1035 Longworth House Office Building, Washingt...","170 Cannon House Office Building, Washington, ..."
Phone,(202) 225-3261,(202) 225-2665,(202) 225-4921


In [12]:
ideology = ideology[['icpsr', 'bioguide_id', 'fec_id', 'left_right_ideology']].groupby('icpsr')['left_right_ideology'].mean()

In [13]:
ideology

icpsr
14226    0.363
14435   -0.518
14854    0.334
14858   -0.356
14863    0.184
         ...  
49308   -0.357
49703    0.124
91980    0.384
92336    0.152
99912    0.403
Name: left_right_ideology, Length: 557, dtype: float64

In [17]:
terms = terms.drop(['startYear', 'endYear'], axis = 1)

In [ ]:
def pandas_df_to_dbml(df: pd.DataFrame, table_name: str) -> str:
    """
    Converts a pandas DataFrame to a DBML string.

    Args:
        df: The pandas DataFrame to convert.
        table_name: The name of the table in the DBML schema.

    Returns:
        A DBML string representing the DataFrame schema.
    """

    dbml_string = f"Table {table_name} {{\n"

    for column_name, column_type in df.dtypes.items():
        dbml_type = map_pandas_dtype_to_dbml_type(column_type)
        dbml_string += f"  {column_name} {dbml_type}\n"

    dbml_string += "}\n"
    return dbml_string

def map_pandas_dtype_to_dbml_type(dtype) -> str:
    """Maps a pandas dtype to a DBML type."""
    dtype_name = str(dtype)
    if "int" in dtype_name:
      return "int"
    if "float" in dtype_name:
      return "float"
    if "datetime" in dtype_name:
        return "datetime"
    return "varchar"

In [ ]:
print(pandas_df_to_dbml(bioinfo, 'members'))

Table members {
  bioguide_id varchar
  Full name varchar
  Chamber varchar
  State varchar
  Party varchar
  District float
  birthYear varchar
  image varchar
  Office address varchar
  Phone varchar
  Website varchar
}



In [ ]:
members = pd.merge(bioinfo, ideology,
                   on = 'bioguide_id',
                   how ='outer',
                   indicator = 'matched')

In [ ]:
members